### Modelo de forecasting de demanda SKU 455151

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from xgboost import XGBRegressor
import joblib
import os
import warnings

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'xgboost'

#### 1. Carga de Datos y Filtrado

In [ ]:
# Cargar datos históricos del SKU
df = pd.read_csv('DATA/raw/ventashistoricas_455151.csv')

# Filtrar por país: Colombia
df = df[df['Country'] == 'Colombia']

# Configurar índice de tiempo
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df = df.sort_values('Transaction_Date').set_index('Transaction_Date')

print(f"Registros cargados para Colombia: {df.shape[0]}")
df.head()

#### 2. Ingeniería de Features

In [ ]:
# Crear variables de calendario
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Variables de rezago (Lags) y estadísticas móviles
df['lag_1'] = df['Quantity'].shift(1)
df['lag_7'] = df['Quantity'].shift(7)
df['rolling_mean_7'] = df['Quantity'].shift(1).rolling(window=7).mean()

# Eliminar filas con NaNs resultantes de los lags
df = df.dropna()
df.head()

#### 3. División de Datos (Train / Test Split)

In [ ]:
features = ['day_of_week', 'month', 'is_weekend', 'lag_1', 'lag_7', 'rolling_mean_7']
X = df[features]
y = df['Quantity']

# Separar los últimos 30 días para test
X_train, X_test = X.iloc[:-30], X.iloc[-30:]
y_train, y_test = y.iloc[:-30], y.iloc[-30:]

print(f"Tamaño de entrenamiento: {X_train.shape[0]} días")
print(f"Tamaño de prueba: {X_test.shape[0]} días")

#### 4. Validación Cruzada Temporal (TimeSeriesSplit)

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true)))

print("--- Evaluación con TimeSeriesSplit (5 Folds) ---")

4.1 Modelo Baseline: Promedio Móvil de 7 días

In [ ]:
baseline_preds = X_test['rolling_mean_7']
mae_base = mean_absolute_error(y_test, baseline_preds)
rmse_base = root_mean_squared_error(y_test, baseline_preds)
mape_base = mape(y_test, baseline_preds)

print(f"[Baseline] MAE: {mae_base:.2f} | RMSE: {rmse_base:.2f} | MAPE: {mape_base:.2f}")

4.2 Modelo 2: Random Forest Regressor

In [ ]:
rf_maes, rf_rmses = [], []
for train_idx, val_idx in tscv.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_tr, y_tr)
    preds = rf.predict(X_val)
    
    rf_maes.append(mean_absolute_error(y_val, preds))
    rf_rmses.append(root_mean_squared_error(y_val, preds))

print(f"[Random Forest CV] MAE: {np.mean(rf_maes):.2f} | RMSE: {np.mean(rf_rmses):.2f}")

4.3 Modelo 3: XGBoost Regressor

In [ ]:
xgb_maes, xgb_rmses = [], []
for train_idx, val_idx in tscv.split(X_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
    xgb.fit(X_tr, y_tr)
    preds = xgb.predict(X_val)
    
    xgb_maes.append(mean_absolute_error(y_val, preds))
    xgb_rmses.append(root_mean_squared_error(y_val, preds))

print(f"[XGBoost CV] MAE: {np.mean(xgb_maes):.2f} | RMSE: {np.mean(xgb_rmses):.2f}")

#### 5. Entrenamiento Final y Exportación

In [ ]:
# Entrenar modelo final con todos los datos de entrenamiento disponibles
xgb_final = XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
xgb_final.fit(X_train, y_train)

# Evaluar en el conjunto de prueba (Test)
final_preds = xgb_final.predict(X_test)
print(f"--- Rendimiento Final XGBoost en Test ---")
print(f"MAE: {mean_absolute_error(y_test, final_preds):.2f}")
print(f"RMSE: {root_mean_squared_error(y_test, final_preds):.2f}")
print(f"MAPE: {mape(y_test, final_preds):.2f}")

# Guardar el artefacto del modelo entrenado
os.makedirs('./models', exist_ok=True)
joblib.dump(xgb_final, './models/modelo_455151_colombia.pkl')
print("¡Modelo guardado exitosamente en ./models/modelo_455151_colombia.pkl!")